In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import shutil

folder = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)

original = folder / "eta_dataset_final.csv"
backup = folder / "eta_dataset_final_backup_before_feature_reduction.csv"

# 1. Read the original dataset and preserve a backup
df = pd.read_csv(original, low_memory=False)

if not backup.exists():
    shutil.copy2(original, backup)

# 2. Select ETA-relevant columns that exist in your dataset
candidate_features = [
    "request_id",
    "urgency_label",
    "people_affected",
    "people_trapped",
    "injuries_reported",
    "medical_emergency",
    "water_level",
    "building_condition",
    "rescue_required",
    "location_risk",
    "context_peak_flood_level_m",
    "context_event_duration_days",
    "context_rainfall_t1d",
    "context_percent_flooded_area",
    "context_dfsi",
    "flood_type",
    "peak_discharge_q_cumec",
    "flood_volume_cumec",
    "time_to_peak_days",
    "recession_time_day",
    "drainage_area",
    "catchment_relief",
    "drainage_density",
    "annual_precipitation",
    "road_density",
    "population_density",
    "urban_percentage",
]

selected = [c for c in candidate_features if c in df.columns]
eta = df[selected].copy()

if "request_id" not in eta.columns:
    raise ValueError("request_id is missing; original file was not changed.")

# 3. Clean selected feature values
for col in eta.select_dtypes(include="number").columns:
    eta[col] = eta[col].replace([np.inf, -np.inf], np.nan)
    if eta[col].notna().any():
        eta[col] = eta[col].fillna(eta[col].median())

for col in eta.select_dtypes(
    include=["object", "string", "category"]
).columns:
    eta[col] = eta[col].fillna("Unknown").astype(str).str.strip()
    eta[col] = eta[col].replace("", "Unknown")

# 4. Preserve existing synthetic ETA labels when available.
# Generate labels only if the original target is absent.
if "actual_travel_minutes" in df.columns:
    eta["actual_travel_minutes"] = pd.to_numeric(
        df["actual_travel_minutes"], errors="coerce"
    )
else:
    rng = np.random.default_rng(42)
    distance = rng.uniform(2, 20, len(df))
    urgency = (
        eta["urgency_label"].astype(str).str.lower()
        if "urgency_label" in eta.columns
        else pd.Series("medium", index=eta.index)
    )
    adjustment = urgency.map({
        "critical": -2, "high": -1, "medium": 0, "low": 2
    }).fillna(0)
    eta["actual_travel_minutes"] = np.round(
        (distance / 25 * 60 + adjustment
         + rng.normal(0, 2, len(df))).clip(3, 120),
        1
    )

eta["eta_label_source"] = "synthetic_simulation"
eta["eta_is_synthetic"] = 1

# 5. Validate before overwriting the original
assert len(eta) == len(df) == 5016
assert eta["request_id"].is_unique
assert eta["actual_travel_minutes"].notna().all()

# 6. Replace original file with the reduced dataset
eta.to_csv(original, index=False)

# 7. Verify saved file
check = pd.read_csv(original, low_memory=False)

print("Original final dataset updated successfully.")
print("Rows:", len(check))
print("Columns:", len(check.columns))
print("Unique requests:", check["request_id"].nunique())
print("Missing cells:", int(check.isna().sum().sum()))
print("Backup:", backup)
print("Updated file:", original)


Original final dataset updated successfully.
Rows: 5016
Columns: 30
Unique requests: 5016
Missing cells: 0
Backup: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final_backup_before_feature_reduction.csv
Updated file: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv


In [2]:

from pathlib import Path
import pandas as pd

path = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv"
)

df = pd.read_csv(path, low_memory=False)

print("Current shape:", df.shape)
print("\nCurrent columns:")
for col in df.columns:
    print(col)

print("\nETA-related summary:")
for col in [
    "actual_travel_minutes",
    "eta_label_source",
    "eta_is_synthetic",
    "synthetic_route_distance_km",
    "urgency_label",
    "water_level",
]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].head(5).to_list())


Current shape: (5016, 30)

Current columns:
request_id
urgency_label
people_affected
people_trapped
injuries_reported
medical_emergency
water_level
building_condition
rescue_required
location_risk
context_peak_flood_level_m
context_event_duration_days
context_rainfall_t1d
context_percent_flooded_area
context_dfsi
flood_type
peak_discharge_q_cumec
flood_volume_cumec
time_to_peak_days
recession_time_day
drainage_area
catchment_relief
drainage_density
annual_precipitation
road_density
population_density
urban_percentage
actual_travel_minutes
eta_label_source
eta_is_synthetic

ETA-related summary:

actual_travel_minutes:
[41.1, 31.0, 47.4, 41.5, 12.0]

eta_label_source:
['synthetic_simulation', 'synthetic_simulation', 'synthetic_simulation', 'synthetic_simulation', 'synthetic_simulation']

eta_is_synthetic:
[1, 1, 1, 1, 1]

urgency_label:
['Critical', 'Critical', 'Critical', 'Critical', 'High']

water_level:
['Rapidly_Rising', 'Rising', 'Rapidly_Rising', 'Rising', 'Rising']


In [3]:

from pathlib import Path
import pandas as pd
import numpy as np
import shutil

folder = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)

file = folder / "eta_dataset_final.csv"
backup = folder / "eta_dataset_backup_before_eta_fix.csv"

# Load and back up
df = pd.read_csv(file, low_memory=False)

if not backup.exists():
    shutil.copy2(file, backup)

if len(df) != 5016 or not df["request_id"].is_unique:
    raise ValueError("Unexpected row count or duplicate request IDs.")

# Generate reproducible simulated route distance.
# This is an assumption, not a measured route.
rng = np.random.default_rng(42)
distance_km = rng.uniform(2, 20, len(df))

# Urgency adjustment (minutes)
urgency = df["urgency_label"].astype(str).str.lower()
urgency_adjustment = urgency.map({
    "critical": -2.0,
    "high": -1.0,
    "medium": 0.0,
    "low": 2.0
}).fillna(0.0).to_numpy()

# Simulated flood-related delay (minutes)
flood_delay = np.zeros(len(df))

water = df["water_level"].astype(str).str.lower()
flood_delay += np.where(
    water.str.contains("rapid|rising", na=False), 5.0, 0.0
)

building = df["building_condition"].astype(str).str.lower()
flood_delay += np.where(
    building.str.contains("damaged|collaps", na=False), 3.0, 0.0
)

# Assume 25 km/h baseline speed; add modest simulation noise.
# All resulting values remain synthetic.
base_minutes = distance_km / 25.0 * 60.0
noise = rng.normal(0, 2, len(df))

eta_minutes = np.clip(
    base_minutes + urgency_adjustment + flood_delay + noise,
    3,
    120
)

# Update columns together
updates = pd.DataFrame({
    "synthetic_route_distance_km": np.round(distance_km, 2),
    "actual_travel_minutes": np.round(eta_minutes, 1),
    "eta_label_source": "synthetic_simulation",
}, index=df.index)

# Remove redundant synthetic flag if present
df = df.drop(columns=["eta_is_synthetic"], errors="ignore")

# Replace old target/source and add simulated distance
df = df.drop(
    columns=[
        "actual_travel_minutes",
        "eta_label_source",
        "synthetic_route_distance_km",
    ],
    errors="ignore",
)

df = pd.concat([df, updates], axis=1).copy()

# Validate
assert len(df) == 5016
assert df["request_id"].is_unique
assert df["actual_travel_minutes"].notna().all()
assert df["synthetic_route_distance_km"].between(2, 20).all()
assert df["actual_travel_minutes"].between(3, 120).all()

# Save to the same final filename
df.to_csv(file, index=False)

print("Corrected ETA dataset saved:", file)
print("Shape:", df.shape)
print("Unique requests:", df["request_id"].nunique())
print("Missing values:", int(df.isna().sum().sum()))
print("ETA source:", df["eta_label_source"].value_counts().to_dict())
print("\nETA summary:")
print(df["actual_travel_minutes"].describe())
print("\nNew columns:", [
    "synthetic_route_distance_km",
    "actual_travel_minutes",
    "eta_label_source",
])


Corrected ETA dataset saved: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv
Shape: (5016, 30)
Unique requests: 5016
Missing values: 0
ETA source: {'synthetic_simulation': 5016}

ETA summary:
count    5016.000000
mean       29.718062
std        12.902169
min         3.000000
25%        18.800000
50%        29.600000
75%        40.700000
max        58.300000
Name: actual_travel_minutes, dtype: float64

New columns: ['synthetic_route_distance_km', 'actual_travel_minutes', 'eta_label_source']


In [4]:

import pandas as pd

file_path = r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv"

# Load the dataset
df = pd.read_csv(file_path)

# Remove the column
df = df.drop(columns=["eta_label_source"], errors="ignore")

# Save to the same file
df.to_csv(file_path, index=False)

print("eta_label_source column removed successfully!")
print("Updated dataset shape:", df.shape)
print("Remaining columns:")
print(df.columns.tolist())


eta_label_source column removed successfully!
Updated dataset shape: (5016, 29)
Remaining columns:
['request_id', 'urgency_label', 'people_affected', 'people_trapped', 'injuries_reported', 'medical_emergency', 'water_level', 'building_condition', 'rescue_required', 'location_risk', 'context_peak_flood_level_m', 'context_event_duration_days', 'context_rainfall_t1d', 'context_percent_flooded_area', 'context_dfsi', 'flood_type', 'peak_discharge_q_cumec', 'flood_volume_cumec', 'time_to_peak_days', 'recession_time_day', 'drainage_area', 'catchment_relief', 'drainage_density', 'annual_precipitation', 'road_density', 'population_density', 'urban_percentage', 'synthetic_route_distance_km', 'actual_travel_minutes']
